In [ ]:
import os, warnings
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # run from the project root so data/ paths resolve
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from cifresignificative.analysis.data import *
from cifresignificative.viz.style import *
apply_style()
panel = load_panel()


# 08 - Productivity by sector
Real gross value added per hour worked (Eurostat national accounts, chain-linked 2020 euro), market sectors only (real estate excluded because of imputed rents). Only growth is compared across countries: euro levels would need PPPs.

In [ ]:
sectors = load_sectors()
agg = aggregate_productivity(sectors)
cover = agg[[c for c in agg if c in EU27_ISO3]].notna().sum(axis=1)
agg = agg.loc[:cover[cover >= 20].index.max()]  # stop at the last year with broad country coverage
idx = agg / agg.loc[2005] * 100
fig, ax = plt.subplots(figsize=(7.5, 4))
for c in EU27_ISO3:
    if c != ITALY and c in idx: ax.plot(idx.index, idx[c], color=PEER_C, lw=0.7, alpha=0.7)
for c, lab in [("DEU", "Germany"), ("FRA", "France"), ("ESP", "Spain")]:
    ax.annotate(lab, (idx[c].dropna().index.max(), idx[c].dropna().iloc[-1]), xytext=(4, 0), textcoords="offset points", fontsize=8, color=INK2, va="center")
med = idx[[c for c in idx if c in EU27_ISO3 and c != ITALY]].median(axis=1)
ax.plot(med.index, med, color=EU_C, lw=2, ls="--"); ax.plot(idx.index, idx[ITALY], color=ITALY_C, lw=2.5)
ax.annotate("Italy", (idx[ITALY].dropna().index.max(), idx[ITALY].dropna().iloc[-1]), xytext=(4, 0), textcoords="offset points", color=ITALY_C, fontweight="bold", va="center")
lo, hi = np.nanpercentile(idx.values, [1, 99]); ax.set_ylim(lo - 5, hi + 5)
ax.set_title("Real output per hour, market sectors (2005 = 100)"); ax.set_ylabel("index"); save(fig, "08_productivity_index")
y1 = int(agg[ITALY].dropna().index.max()); print("Last year:", y1, "| Italy:", round(idx.loc[y1, ITALY], 1), "| median of others:", round(med.loc[y1], 1), "| DEU/FRA/ESP:", idx.loc[y1, ["DEU", "FRA", "ESP"]].round(1).to_dict())

In [ ]:
# Sector productivity growth, 2005-2019 (average annual log growth), Italy vs the median of the other members
def sector_growth(a, b):
    p = sectors.pivot_table(index="iso3", columns=["nace_r2", "year"], values="prod")
    return pd.DataFrame({n: np.log(p[(n, b)] / p[(n, a)]) / (b - a) * 100 for n in MARKET_SECTORS})
g = sector_growth(2005, 2019).loc[[c for c in EU27_ISO3 if c in sectors.iso3.unique()]]
tab = pd.DataFrame({"Italy": g.loc[ITALY], "median of others": g.drop(ITALY).median(), "Germany": g.loc["DEU"]}).rename(index=SECTOR_LABELS).sort_values("Italy")
fig, ax = plt.subplots(figsize=(8, 4.2)); yy = np.arange(len(tab))
ax.scatter(tab["median of others"], yy, color=EU_C, s=40, zorder=3, label="median of other EU27", marker="D")
ax.scatter(tab["Germany"], yy, color=PEER_C, s=36, zorder=2, label="Germany")
ax.barh(yy, tab["Italy"], color=ITALY_C, height=0.45, zorder=2, label="Italy")
ax.set_yticks(yy, tab.index); ax.axvline(0, color=INK2, lw=0.8); ax.set_xlabel("% per year, real value added per hour, 2005-2019")
ax.set_title("Sector productivity growth, 2005-2019"); ax.legend(fontsize=8, loc="lower right"); save(fig, "08_sector_growth"); display(tab.round(2))

In [ ]:
# Shift-share: within-sector productivity vs reallocation of hours across sectors (percent change in aggregate productivity)
countries = ["ITA", "DEU", "FRA", "ESP", "PRT", "GRC"]
rows = {c: shift_share(sectors, c, 2005, 2019) for c in countries}
oth = pd.DataFrame({c: shift_share(sectors, c, 2005, 2019) for c in EU27_ISO3 if c in sectors.iso3.unique() and c != ITALY}).T.median()
dec = pd.DataFrame(rows).T; dec.loc["EU median (excl. IT)"] = oth
fig, ax = plt.subplots(figsize=(8, 3.8)); cols = ["#2a78d6", "#eda100", "#b9b8b2"]
pos = np.zeros(len(dec)); neg = np.zeros(len(dec))
for part, col in zip(dec.columns, cols):
    v = dec[part].values; base = np.where(v >= 0, pos, neg)
    ax.barh(range(len(dec)), v, left=base, color=col, height=0.6, label=part, edgecolor=SURFACE, linewidth=2)
    pos += np.where(v >= 0, v, 0); neg += np.where(v < 0, v, 0)
ax.scatter(dec.sum(axis=1), range(len(dec)), color=INK, s=26, zorder=5, label="total")
ax.set_yticks(range(len(dec)), dec.index); ax.invert_yaxis(); ax.axvline(0, color=INK2, lw=0.8)
ax.set_xlabel("% change in aggregate output per hour, 2005-2019"); ax.set_title("Where productivity growth came from, 2005-2019"); ax.legend(fontsize=8, loc="lower right"); save(fig, "08_shift_share")
display(dec.round(1))

In [ ]:
# Structure: share of hours by sector, 2005 vs latest year with data (Italy vs median of the others)
def hshare(y):
    h = sectors.pivot_table(index="iso3", columns=["nace_r2", "year"], values="hours_ths")
    t = pd.DataFrame({n: h[(n, y)] for n in MARKET_SECTORS}); return t.div(t.sum(axis=1), axis=0) * 100
a, b = hshare(2005), hshare(2022)
out = pd.DataFrame({"Italy 2005": a.loc[ITALY], "Italy 2022": b.loc[ITALY], "others 2005 (median)": a.drop(ITALY).median(), "others 2022 (median)": b.drop(ITALY).median()}).rename(index=SECTOR_LABELS)
display(out.round(1))

## Takeaways (Eurostat national accounts; market sectors excl. real estate; 2005-2019 unless stated)
- **Aggregate output per hour barely moved:** index 2005 = 100 reaches 102 in 2024 for Italy against 124 for the median of the other members (Germany 113, France 108, Spain 115).
- **The gap is within sectors, not between them.** Over 2005-2019 within-sector productivity added +2.8% for Italy vs +16.9% for the median (Germany +12.8, France +9.9, Spain +15.4); the reallocation of hours across the nine broad sections was -0.5 for Italy vs +2.7 median. A shift of hours between sections is therefore not what explains the gap. This view cannot see misallocation across *firms within* a section, which is where the literature (Hsieh-Klenow; Gopinath et al.) places it: that needs firm-level data.
- **Italy trails the median in seven of nine sections.** Annual growth in output per hour: construction -1.4% (median +0.4%), professional & administrative services -1.3% (+0.7%), information & communication +0.7% (+1.9%; Germany +3.6%), industry +0.9% (+1.8%), agriculture +0.5% (+2.7%). Finance is on par (+1.4%). Public administration, education and health is -0.6% (median +0.1%).
- Structure: industry fell from 20.6% to 17.5% of hours (2005-2022) while professional & administrative services rose from 9.9% to 13.4%, a sector where measured productivity fell. Deflators in services are harder to measure, so sector rankings should be treated with care.